# Consolidate all files in the various directories into a single data frame

## Setup

In [ ]:
import re
from pathlib import Path
from datasets import load_dataset, VerificationMode
from collections import defaultdict
from typing import Any, Literal

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
base = Path(".").resolve() / "eval-all" / "data"
assert base.is_dir() and base.exists()
base

## Load the dataset

In [ ]:
# Somehow this is necessary
splits = ["train", "val", "test"]
options = dict(verification_mode=VerificationMode.NO_CHECKS)

# Only download these splits
# splits = ["val", "test"]
# options = dict(
#     **options, data_files={split: f"{split}-*" for split in splits},
# )

ds_binary = load_dataset(
    "dasyd/time-qa",
    "binary",
    data_dir="binary",
    **options,
)
ds_multi = load_dataset(
    "dasyd/time-qa",
    "multi",
    data_dir="multi",
    **options,
)
ds_open = load_dataset(
    "dasyd/time-qa",
    "open",
    data_dir="open",
    **options,
)

In [ ]:
concat_dataset = (
    pd.concat(
        [
            ds.remove_columns("trajectory").to_pandas()
            for ds_dict in [ds_binary, ds_multi, ds_open]
            for ds in ds_dict.values()
        ],
        ignore_index=True,
    )
    .set_index(["sample_id", "question_id"])
    .sort_index()
)
concat_dataset

In [ ]:
concat_dataset.info()

## Load experimental data

In [ ]:
def split_compound_id(df: pd.DataFrame) -> pd.DataFrame:
    """Takes a dataframe and splits the 'id' column into 'sample_id' and 'question_id'."""

    df = df.copy()
    original_id = df["id"].astype(str)

    del df["id"]
    df["sample_id"] = original_id.str.slice(0, -1).astype(int)
    df["question_id"] = original_id.str.slice(-1).astype(int)

    return df

In [ ]:
# split_compound_id(pd.read_csv(base / "4.2.3" / "llama3.1_fewshot-prompt_test.csv"))

In [ ]:
# Keys are (setting/experiment, model, mode)
aggregated: dict[tuple[str, str, str], pd.DataFrame] = dict()

for setting in base.iterdir():
    # setting must start with number dot number (check with regex)
    if not setting.is_dir() or re.match(r"^\d+\.\d+", setting.name) is None:
        print(f"SKIPPING {setting}")
        continue

    for experiment in setting.iterdir():
        if experiment.is_dir():
            print(f"SKIPPING directory {experiment}")
            continue
        if experiment.suffix != ".csv":
            continue

        name = experiment.stem
        components = name.rsplit("_", 2)
        if len(components) != 3:
            print(f"SKIPPING invalid name {name} in '{experiment}'")
            continue

        model, mode, split = components

        # Mode is either "full" or an integer
        try:
            int(mode)
        except ValueError:
            match mode:
                case "full":
                    pass
                case "fewshot-prompt":
                    if setting.name in ("4.2.3", "4.2.4"):
                        # Special case for fewshot-prompt in 4.2.3 and 4.2.4
                        mode = 50  # TODO check if correct!
                    else:
                        print(f"SKIPPING special case {mode} in '{experiment}'")
                        continue
                case _:
                    print(f"SKIPPING invalid mode {mode} in '{experiment}'")
                    continue

        assert split in ("val", "test"), f"Invalid split {split} in '{experiment}'"

        print(f"Setting {setting.name}, model {model}, mode {mode}, split {split}")
        data = pd.read_csv(experiment)
        assert "prediction_text" in data.columns, (
            f"Missing prediction_text in '{experiment}'"
        )

        if "id" in data.columns:
            data = split_compound_id(data)
        assert "sample_id" in data.columns, f"Missing sample_id in '{experiment}'"
        assert "question_id" in data.columns, f"Missing question_id in '{experiment}'"

        assert "answer_text" not in data.columns

        aggregated[(setting.name, model, mode)] = data[
            ["sample_id", "question_id", "prediction_text"]
        ]

In [ ]:
len(aggregated)

In [ ]:
df = pd.concat(aggregated, verify_integrity=True, copy=False).reset_index(
    level=3, drop=True
)
df.index.names = ["setting", "model", "mode"]
df.reset_index(inplace=True)
df.set_index(["setting", "model", "mode", "sample_id", "question_id"], inplace=True)
df.sort_index(inplace=True)
df.reset_index(inplace=True)
df

In [ ]:
df.info()

In [ ]:
df["setting"].value_counts()

## Join and clean them

In [ ]:
joined = df.join(
    concat_dataset, validate="many_to_one", on=["sample_id", "question_id"]
)
joined

In [ ]:
# array(['falcon', 'falcon2', 'gemma2_2b', 'gemma2_9b', 'gemma_2b',
#        'gemma_7b', 'llama2', 'llama3', 'llama3.1', 'mistral', 'mvp',
#        't5_base', 't5_small', 'umt5_base', 'umt5_small'], dtype=object)

joined["model"] = joined["model"].map(
    {
        "falcon": "Falcon (7B)",
        "falcon2": "Falcon 2 (11B)",
        "gemma_2b": "Gemma (2B)",
        "gemma_7b": "Gemma (7B)",
        "gemma2_2b": "Gemma 2 (2B)",
        "gemma2_9b": "Gemma 2 (9B)",
        "llama2": "Llama 2 (8B)",
        "llama3": "Llama 3 (8B)",
        "llama3.1": "Llama 3.1 (8B)",
        "mistral": "Mistral (7B)",
        "mvp": "MVP",
        "t5_base": "T5 Base",
        "t5_small": "T5 Small",
        "umt5_base": "UMT5 Base",
        "umt5_small": "UMT5 Small",
    }
)

In [ ]:
pd.Series(joined["model"].unique()).sort_values()

In [ ]:
joined["answer_type"].value_counts(dropna=False)

In [ ]:
# Cut away potential whitespace and trailing "\n\n#" and following characters from the prediction text
joined["prediction_text"] = (
    joined["prediction_text"]
    .astype(str)
    .str.strip()
    .str.split(r"([\r\n]|\\[rn])+#", n=1, regex=True)
    .str[0]
    .str.strip()
    .str.replace("<0x0A>", "")
    .replace("nan", pd.NA)
)

### Binary

In [ ]:
# The ones we were not able to parse are:
joined.loc[
    (joined["answer_type"] == "multi"),
    "prediction_text",
].value_counts(dropna=False)[:50]

In [ ]:
joined.loc[joined["answer_type"] == "binary", "prediction_parsed_binary"] = (
    joined.loc[joined["answer_type"] == "binary", "prediction_text"]
    .str.lower()
    # find the first "word" (sentence, really):
    .str.split(r"(\\)|\s|[.,]", n=1, regex=True)
    .str[0]
    .map(
        defaultdict(
            lambda: pd.NA,
            {
                "true": True,
                "yes": True,
                "yes.": True,
                "correct": True,
                "false": False,
                "no": False,
                "no.": False,
                "incorrect": False,
            },
        )
    )
)

# We parsed:
joined.loc[joined["answer_type"] == "binary", "prediction_parsed_binary"].value_counts(
    dropna=False
)

In [ ]:
ax = sns.displot(
    data=joined[joined["answer_type"] == "binary"].astype(
        {"prediction_parsed_binary": "str"}
    ),
    y="model",
    col="setting",
    col_wrap=2,
    hue="prediction_parsed_binary",
    multiple="stack",
)
sns.move_legend(ax, bbox_to_anchor=(1.05, 1), loc="upper left")
sns.despine()
pass

In [ ]:
# The ones we were not able to parse are:
joined.loc[
    (joined["answer_type"] == "binary") & (joined["prediction_parsed_binary"].isna()),
    "prediction_text",
].value_counts(dropna=False)

### Multi

In [ ]:
joined[joined["answer_type"] == "multi"]["prediction_text"].value_counts(dropna=False)[
    :50
]

In [ ]:
joined[joined["answer_type"] == "multi"]["options"][:5]

In [ ]:
joined[joined["answer_type"] == "multi"]["prediction_text"]

In [ ]:
def parse(row: dict[str, Any]) -> str | Literal[pd.NA]:
    if pd.isna(row["prediction_text"]):
        return pd.NA

    value = row["prediction_text"].upper()

    match value:
        case "A" | "B" | "C":
            return value
    # else, we need to parse the options:

    # options are a dict, we want to find the key to the closest value match
    options = row["options"]
    if not options:
        raise ValueError(f"Missing options for {row}")

    # closest_key = min(options, key=lambda k: abs(len(value) - len(options[k])))

    # Now, we use exact matching to get the key
    return next((k for k, v in options.items() if v.upper() == value), pd.NA)


# Derive a new parsed column from each row
joined.loc[joined["answer_type"] == "multi", "prediction_parsed_multi"] = joined[
    joined["answer_type"] == "multi"
].apply(parse, axis="columns")

# We parsed:
joined.loc[joined["answer_type"] == "multi", "prediction_parsed_multi"].value_counts(
    dropna=False
)

In [ ]:
ax = sns.displot(
    data=joined[(joined["answer_type"] == "multi")].astype(
        {"prediction_parsed_multi": "str"}
    ),
    y="model",
    col="setting",
    col_wrap=2,
    hue="prediction_parsed_multi",
    multiple="stack",
)
sns.move_legend(ax, bbox_to_anchor=(1.05, 1), loc="upper left")
sns.despine()
pass

In [ ]:
# Check UMT5 Small (4.2.1)

joined_with_missing = joined.copy()
joined_with_missing = joined_with_missing[joined_with_missing["setting"] == "4.2.1"]
joined_with_missing = joined_with_missing[joined_with_missing["model"] == "UMT5 Small"]
joined_with_missing["prediction_text_is_missing"] = joined_with_missing[
    "prediction_text"
].isna()

ax = sns.displot(
    data=joined_with_missing.astype({"prediction_text_is_missing": "str"}),
    y="mode",
    hue="prediction_text_is_missing",
    multiple="stack",
)
ax.set(title="Missing predictions in UMT5 Small (4.2.1)")
sns.despine()
pass

In [ ]:
# The ones we were not able to parse are:
joined.loc[
    (joined["answer_type"] == "multi") & (joined["prediction_parsed_multi"].isna()),
    "prediction_text",
].value_counts(dropna=False)[:50]

### Open

In [ ]:
joined[joined["answer_type"] == "open"]["prediction_text"].value_counts(dropna=False)[
    :50
]

# Store them

In [ ]:
joined.info()

In [ ]:
# joined.to_parquet(base / "all_joined.parquet")  # Somehow gave an error
joined.to_hdf(base / "all_joined.h5", key="data", mode="w")